# PC Offload Colab Bootstrap

Use this notebook as a remote VM extension of the local machine. It mounts Google Drive, syncs the bootstrap repo, and gives you a safe place to run heavy work without loading local RAM or disk.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/israelrealivazquez-lang/pc-speed-cloud-bootstrap.git'
WORKDIR = Path('/content/pc-speed-cloud-bootstrap')
DRIVE_ROOT = Path('/content/drive/MyDrive/Antigravity_Cloud_Migration')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if WORKDIR.exists():
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)

print('Repo ready at', WORKDIR)
print('Drive root:', DRIVE_ROOT)

In [ ]:
import os

targets = [
    DRIVE_ROOT / 'Offload_From_Dropbox_2026-04-19',
    DRIVE_ROOT / 'Offload_From_Profile_2026-04-19',
    DRIVE_ROOT / 'Offload_From_OneDrive_2026-04-19'
]

for target in targets:
    if target.exists():
        total = 0
        count = 0
        for root, _, files in os.walk(target):
            for name in files:
                path = Path(root) / name
                try:
                    total += path.stat().st_size
                    count += 1
                except OSError:
                    pass
        print(f'{target}: {count} files, {round(total / 1e9, 3)} GB')
    else:
        print(f'{target}: missing')

In [ ]:
from pathlib import Path
import shutil

def stage_from_drive(relative_path: str, local_root: str = '/content/work'):
    src = DRIVE_ROOT / relative_path
    dst = Path(local_root) / Path(relative_path).name
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        if dst.is_dir():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    return dst

def persist_to_drive(local_path: str, relative_path: str):
    src = Path(local_path)
    dst = DRIVE_ROOT / relative_path
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        if dst.is_dir():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    return dst

print('Helpers loaded: stage_from_drive(), persist_to_drive()')

## Suggested usage

1. Mount Drive.
2. Pull the latest repo version.
3. Stage a large folder from Drive into `/content/work`.
4. Run heavy processing in Colab.
5. Persist results back into `MyDrive/Antigravity_Cloud_Migration`.